# 09 — F1 y MAE ponderados: comparación 03 vs 05

Recalcula las métricas de los notebooks 03 y 05 usando **F1 y MAE ponderados por soporte** (número real de casos positivos de cada clase) en vez de la media macro simple. No se reentrena ningún modelo: se cargan los `.keras` ya guardados y se reutiliza el caché de tensores STFT (`con_filtrado`), así que solo hace falta reconstruir `X_val`, `Y_val`, `X_test` y `Y_test` — no `X_train`, que no se necesita para evaluar.

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos
from sklearn.metrics import precision_recall_curve

import tensorflow as tf

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general

Mismo `PREP_VERSION` (`con_filtrado`) que usan los notebooks 02 a 05, así que el caché de tensores STFT ya existe en disco y no se recalcula nada.

In [2]:
DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / "con_filtrado"

MODEL_DIR = Path("./outputs/models")

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)

## 3. Carga de metadatos y etiquetas multi-etiqueta

In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)

## 4. Split por paciente

In [4]:
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

print("Tamaños:", len(val_df), len(test_df))

Tamaños: 2146 2158


## 5. Preprocesado y espectrograma STFT

Idéntico a los notebooks 02-05, carácter por carácter — necesario para que las claves de caché (hash del `rel_path`) coincidan con los archivos `.npy` ya guardados en disco.

In [5]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    """Descarta derivaciones corruptas: con NaN/Inf o completamente planas (fallo de electrodo)."""
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    """Limpia una derivación (1D): paso-banda + notch + normalización z-score."""
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    """Limpia las 12 derivaciones de un ECG. Descarta el registro si más de la mitad fallan."""
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned

def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    """Convierte una derivación 1D en su espectrograma STFT normalizado, con forma fija (F, T)."""
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))

    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad

    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    """signal_2d: (n_muestras, 12) -> tensor (F, T, 12)."""
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)

## 6. Carga de X_val, Y_val, X_test, Y_test desde el caché existente

No se recalcula ningún tensor: si algún ECG no estuviera todavía en caché (no debería pasar, ya que 02-05 ya lo generaron), se calcula sobre la marcha, pero lo normal es que todo esté ya guardado.

In [6]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_val,  Y_val  = build_split(val_df)
X_test, Y_test = build_split(test_df)

print("Formas:", X_val.shape, X_test.shape)

Formas: (2146, 33, 64, 12) (2158, 33, 64, 12)


## 7. Métricas macro y ponderadas

`compute_metrics` es la misma función que en 03 y 05 (macro simple). `f1_mae_ponderado` añade la versión ponderada por soporte real, calculado directamente de `Y_true.sum(axis=0)` — sin estimaciones, el conteo exacto de positivos por clase.

In [7]:
def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

def f1_mae_ponderado(metrics_df, Y_true):
    support = Y_true.sum(axis=0)
    f1_por_clase  = metrics_df.iloc[:5]["F1"].values
    mae_por_clase = metrics_df.iloc[:5]["MAE"].values
    f1_ponderado  = (f1_por_clase * support).sum() / support.sum()
    mae_ponderado = (mae_por_clase * support).sum() / support.sum()
    return support, f1_ponderado, mae_ponderado

## 8. Notebook 03 — carga del modelo y evaluación


In [8]:
model_03 = tf.keras.models.load_model(MODEL_DIR / "03_cnn_profunda_con_filtrado.keras", compile=False)

test_probs_03 = model_03.predict(X_test, verbose=0)
test_pred_03  = (test_probs_03 >= 0.5).astype(int)

metrics_03 = compute_metrics(Y_test, test_pred_03, test_probs_03)
metrics_03

,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.842447,0.770634,0.921080,0.839167,0.187454
1,Infarto,0.823448,0.792388,0.416364,0.545888,0.211895
2,Isquemia/ST-T,0.854958,0.665079,0.804223,0.728063,0.175597
3,Hipertrofia,0.888323,0.565217,0.347328,0.430260,0.155766
4,Conducción (bloqueos),0.862373,0.875472,0.467742,0.609724,0.163442
5,Macro,0.854310,0.733758,0.591347,0.630620,0.178831


## 9. Notebook 05 — carga del modelo, umbrales por clase y evaluación

El 05 no usa umbral fijo 0.5: los umbrales óptimos por clase se recalculan igual que en su notebook original, sobre `X_val`/`Y_val` con `precision_recall_curve`.

In [9]:
model_05 = tf.keras.models.load_model(MODEL_DIR / "05_cnn_profunda_balanceo_umbral.keras", compile=False)

val_probs_05 = model_05.predict(X_val, verbose=0)

best_thresholds = {}
for i, c in enumerate(SUPER_CLASSES):
    prec, rec, thr = precision_recall_curve(Y_val[:, i], val_probs_05[:, i])
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-8, None)
    best_i = np.nanargmax(f1[:-1]) if len(thr) > 0 else 0
    best_thresholds[c] = float(thr[best_i]) if len(thr) > 0 else 0.5

print("Umbrales óptimos por clase (validación):", best_thresholds)
THRESH_VEC = np.array([best_thresholds[c] for c in SUPER_CLASSES])

test_probs_05 = model_05.predict(X_test, verbose=0)
test_pred_05  = (test_probs_05 >= THRESH_VEC).astype(int)

metrics_05 = compute_metrics(Y_test, test_pred_05, test_probs_05)
metrics_05

Umbrales óptimos por clase (validación): {'NORM': 0.4524393677711487, 'MI': 0.42697495222091675, 'STTC': 0.6164358854293823, 'HYP': 0.6522840261459351, 'CD': 0.7015767097473145}


,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.831789,0.755973,0.920042,0.829977,0.266208
1,Infarto,0.783596,0.553966,0.774545,0.645944,0.332317
2,Isquemia/ST-T,0.854495,0.661972,0.811900,0.729310,0.295048
3,Hipertrofia,0.869787,0.465704,0.492366,0.478664,0.385338
4,Conducción (bloqueos),0.864690,0.751232,0.614919,0.676275,0.333178
5,Macro,0.840871,0.637769,0.722755,0.672034,0.322418


## 10. Comparación final: macro vs. ponderado

In [10]:
support_03, f1_pond_03, mae_pond_03 = f1_mae_ponderado(metrics_03, Y_test)
support_05, f1_pond_05, mae_pond_05 = f1_mae_ponderado(metrics_05, Y_test)

print("Soporte por clase (igual en ambos, mismo test set):")
print(dict(zip(SUPER_CLASSES, support_03.astype(int))))
print()

resumen = pd.DataFrame([
    {
        "Notebook": "03_cnn_profunda_con_filtrado",
        "F1_macro": metrics_03.loc[5, "F1"], "F1_ponderado": f1_pond_03,
        "MAE_macro": metrics_03.loc[5, "MAE"], "MAE_ponderado": mae_pond_03,
    },
    {
        "Notebook": "05_cnn_profunda_balanceo_umbral",
        "F1_macro": metrics_05.loc[5, "F1"], "F1_ponderado": f1_pond_05,
        "MAE_macro": metrics_05.loc[5, "MAE"], "MAE_ponderado": mae_pond_05,
    },
])

resumen.to_csv(MODEL_DIR / "f1_mae_ponderado_03_05.csv", index=False)
resumen

Soporte por clase (igual en ambos, mismo test set):
{'NORM': 963, 'MI': 550, 'STTC': 521, 'HYP': 262, 'CD': 496}



,Notebook,F1_macro,F1_ponderado,MAE_macro,MAE_ponderado
0,03_cnn_profunda_con_filtrado,0.630620,0.681529,0.178831,0.182817
1,05_cnn_profunda_balanceo_umbral,0.672034,0.714667,0.322418,0.307689
